**The Greeter Engine (Cold Start Model):** Designed to execute at Millisecond Zero before behavioral telemetry is generated. It relies exclusively on Target Encoded categorical features with Bayesian smoothing to predict a user's baseline purchasing propensity the moment they hit the landing page.

In [9]:
import pandas as pd
import warnings
from utils.marketing_eda import plot_correlation_matrix
from utils.marketing_evaluation import evaluate_threshold

# Mute all Python warnings for a pristine notebook output
warnings.filterwarnings('ignore')

# 1. Load the raw data
df = pd.read_csv('../data/raw/online_shoppers_intention.csv')

# 2. Define the "Millisecond Zero" features
cold_start_features = ['VisitorType', 'TrafficType', 'Browser', 'OperatingSystems', 'Weekend']

# 3. Rapid EDA: Print top 5 categories by volume and their conversion rates
for col in cold_start_features:
    print(f"\n--- Top 5 by {col} ---")
    # Group by the feature, calculate volume and mean (conversion rate)
    summary = df.groupby(col)['Revenue'].agg(['count', 'mean']).sort_values(by='count', ascending=False).head(5)
    
    # Format for readability
    summary['mean'] = (summary['mean'] * 100).round(1).astype(str) + '%'
    summary.rename(columns={'count': 'Traffic Volume', 'mean': 'Conversion Rate'}, inplace=True)
    
    print(summary)


--- Top 5 by VisitorType ---
                   Traffic Volume Conversion Rate
VisitorType                                      
Returning_Visitor           10551           13.9%
New_Visitor                  1694           24.9%
Other                          85           18.8%

--- Top 5 by TrafficType ---
             Traffic Volume Conversion Rate
TrafficType                                
2                      3913           21.6%
1                      2451           10.7%
3                      2052            8.8%
4                      1069           15.4%
13                      738            5.8%

--- Top 5 by Browser ---
         Traffic Volume Conversion Rate
Browser                                
2                  7961           15.4%
1                  2462           14.8%
4                   736           17.7%
5                   467           18.4%
6                   174           11.5%

--- Top 5 by OperatingSystems ---
                  Traffic Volume Conversi

In [10]:
import time
import joblib
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, classification_report

print("⚙️ Initializing Greeter Pipeline Construction...")

# 1. Define features and prepare data
features = ['VisitorType', 'TrafficType', 'Browser', 'OperatingSystems', 'Region', 'Month', 'Weekend']
X = df[features]
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Build components inside a trackable layout
preprocessor = ColumnTransformer(transformers=[('target_enc', TargetEncoder(), features)])

# Note: We do NOT pass verbose=-1 to let LightGBM issue its default logs/warnings
greeter_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
        n_estimators=100, 
        learning_rate=0.05, 
        max_depth=5, 
        class_weight='balanced', 
        random_state=42,
        verbose=-1
    ))
])

# 3. Track execution steps explicitly
steps = [
    ("Transforming Categorical Matrix via Target Encoding", lambda: preprocessor.fit(X_train, y_train)),
    ("Optimizing LightGBM Weak Learners (100 Trees)", lambda: greeter_pipeline.fit(X_train, y_train))
]

print("\n🚀 Executing Pipeline Compilation:")
func()


# 4. Evaluate the engine
y_pred_proba = greeter_pipeline.predict_proba(X_test)[:, 1]
y_pred = greeter_pipeline.predict(X_test)

print("\n" + "="*40)
print("🎯 THE GREETER PERFORMANCE (MILLI-SECOND ZERO)")
print("="*40)
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("="*40)

# 5. Freeze artifact
joblib.dump(greeter_pipeline, '../models/greeter_engine_v1.joblib')
print("\n📦 Production artifact sealed: models/greeter_engine_v1.joblib")

⚙️ Initializing Greeter Pipeline Construction...

🚀 Executing Pipeline Compilation:

🎯 THE GREETER PERFORMANCE (MILLI-SECOND ZERO)
ROC-AUC Score: 0.7018

Classification Report:
              precision    recall  f1-score   support

       False       0.92      0.65      0.76      2084
        True       0.26      0.68      0.38       382

    accuracy                           0.65      2466
   macro avg       0.59      0.66      0.57      2466
weighted avg       0.81      0.65      0.70      2466


📦 Production artifact sealed: models/greeter_engine_v1.joblib
